In [4]:

import ollama
import chromadb
import pypdf
import gradio

print("All required libraries imported successfully!")

All required libraries imported successfully!


In [6]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 141 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (506 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [7]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [8]:
import subprocess
import time
import requests

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

for i in range(30):
    try:
        response = requests.get("http://127.0.0.1:11434")
        if response.status_code == 200:
            print("Ollama server is running successfully!")
            break
    except requests.exceptions.ConnectionError:
        time.sleep(2)
else:
    print("Server failed to start. Please check.")

Ollama server is running successfully!


In [9]:
!nvidia-smi

Thu Nov 26 03:48:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [10]:
!ollama pull gemma3:1b

In [11]:
!ollama pull embeddinggemma

In [12]:
!ollama list

NAME                     ID              SIZE      MODIFIED       
embeddinggemma:latest    85462619ee72    621 MB    15 seconds ago    
gemma3:1b                97558784bdc9    806 MB    46 seconds ago    


In [17]:
!ps aux | grep '[o]llama serve'

In [18]:
import subprocess
import time
import requests

# Stop any previous Ollama server
subprocess.run(
    ["pkill", "-f", "ollama serve"],
    capture_output=True
)

time.sleep(2)

# Start Ollama again
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE
)

# Wait for server
for i in range(30):
    try:
        response = requests.get(
            "http://127.0.0.1:11434",
            timeout=2
        )

        if response.status_code == 200:
            print("SUCCESS! Ollama server is running.")
            break

    except requests.exceptions.RequestException:
        time.sleep(2)

else:
    print("Ollama failed to start. Check server logs.")

SUCCESS! Ollama server is running.


In [19]:
!ollama list

NAME                     ID              SIZE      MODIFIED      
embeddinggemma:latest    85462619ee72    621 MB    4 minutes ago    
gemma3:1b                97558784bdc9    806 MB    5 minutes ago    


In [22]:
import ollama

try:
    client = ollama.Client(host='http://127.0.0.1:11434')
    # List available models via the SDK
    models_list = client.list()
    print("Successfully connected to Ollama!")
    print("Available models in SDK:")
    for model in models_list.get('models', []):
        print(f" - {model.get('model')}")

    # Try a quick test run to verify inference works
    print("\nRunning a quick test generation with gemma3:1b...")
    response = client.generate(model='gemma3:1b', prompt='Hello! Say "Ollama is ready!"')
    print(f"Response: {response.get('response')}")
except Exception as e:
    print(f"Error communicating with Ollama: {e}")

Error communicating with Ollama: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download


In [24]:
import ollama

try:
    client = ollama.Client(host='http://127.0.0.1:11434')
    # List available models via the SDK
    models_list = client.list()
    print("Successfully connected to Ollama!")
    print("Available models in SDK:")
    for model in models_list.get('models', []):
        print(f" - {model.get('model')}")

    # Try a quick test run to verify inference works
    print("\nRunning a quick test generation with gemma3:1b...")
    response = client.generate(model='gemma3:1b', prompt='Hello! Say "Ollama is ready!"')
    print(f"Response: {response.get('response')}")
except Exception as e:
    print(f"Error communicating with Ollama: {e}")

Error communicating with Ollama: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download


### Investigating Ollama Server Status
Let's check if the server is still running in the background and read its logs to diagnose the connection failure.

In [25]:
import subprocess

# 1. Check if the process is active
ps_output = subprocess.run(["ps", "aux"], capture_output=True, text=True)
ollama_processes = [line for line in ps_output.stdout.split('\n') if 'ollama' in line]

print("--- Active Ollama Processes ---")
if ollama_processes:
    for proc in ollama_processes:
        print(proc)
else:
    print("No active Ollama process found!")

# 2. Let's start the server in the foreground briefly and capture the logs to see the error
print("\n--- Capturing Ollama Server Startup Logs ---")
try:
    # Run for 3 seconds to capture any startup errors
    proc = subprocess.run(["ollama", "serve"], capture_output=True, text=True, timeout=3)
    print("STDOUT:", proc.stdout)
    print("STDERR:", proc.stderr)
except subprocess.TimeoutExpired as e:
    # If it didn't exit in 3 seconds, it is running or hanging
    print("Process did not crash immediately. Startup Stderr:")
    print(e.stderr if e.stderr else "No immediate stderr logs.")

--- Active Ollama Processes ---
root        8008  0.0  0.0      0     0 ?        Z    03:54   0:00 [ollama] <defunct>

--- Capturing Ollama Server Startup Logs ---
Process did not crash immediately. Startup Stderr:
b'time=2026-11-26T04:00:24.871Z level=INFO source=routes.go:2484 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_CREATE_REMOTE:false OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CL

### Cleaning Up and Restarting Ollama
We will force kill any remaining zombie or defunct processes, launch the background server cleanly, and verify the client connection.

In [26]:
import subprocess
import time
import requests
import ollama

# 1. Forcefully kill any running, zombie, or orphaned ollama processes
print("Cleaning up existing processes...")
subprocess.run(["pkill", "-9", "-f", "ollama"], capture_output=True)
time.sleep(2)

# 2. Launch a completely fresh Ollama server instance in the background
print("Starting fresh background Ollama server...")
server_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# 3. Wait and poll the server until it is fully active
for i in range(15):
    try:
        response = requests.get("http://127.0.0.1:11434", timeout=2)
        if response.status_code == 200:
            print("SUCCESS: Ollama server started cleanly and is fully responsive!")
            break
    except requests.exceptions.RequestException:
        time.sleep(2)
else:
    print("ERROR: Server failed to become responsive.")

# 4. Test communication with the python client SDK
try:
    client = ollama.Client(host='http://127.0.0.1:11434')
    models = client.list()
    print("\nConnection established! Available models:")
    for m in models.get('models', []):
        print(f" - {m.get('model')}")
except Exception as e:
    print(f"Connection test failed: {e}")

Cleaning up existing processes...
Starting fresh background Ollama server...
SUCCESS: Ollama server started cleanly and is fully responsive!

Connection established! Available models:
 - embeddinggemma:latest
 - gemma3:1b


In [27]:
import subprocess
import time
import requests

# Stop any previous Ollama server
subprocess.run(
    ["pkill", "-f", "ollama serve"],
    capture_output=True
)

time.sleep(2)

# Start Ollama again
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE
)

# Wait for server
for i in range(30):
    try:
        response = requests.get(
            "http://127.0.0.1:11434",
            timeout=2
        )

        if response.status_code == 200:
            print("SUCCESS! Ollama server is running.")
            break

    except requests.exceptions.RequestException:
        time.sleep(2)

else:
    print("Ollama failed to start. Check server logs.")

SUCCESS! Ollama server is running.


In [28]:
!ollama list

NAME                     ID              SIZE      MODIFIED       
embeddinggemma:latest    85462619ee72    621 MB    11 minutes ago    
gemma3:1b                97558784bdc9    806 MB    12 minutes ago    


In [29]:
import ollama

client = ollama.Client(
    host="http://127.0.0.1:11434"
)

response = client.chat(
    model="gemma3:1b",
    messages=[
        {
            "role": "user",
            "content": "Explain RAG in three simple sentences."
        }
    ]
)

print(response["message"]["content"])

RAG (Retrieval-Augmented Generation) is a technique that combines a large language model with a knowledge base. It allows the model to access and use external information to generate more accurate and contextually relevant responses. Essentially, it’s like asking a knowledgeable expert instead of just relying on the model’s own memory.


In [30]:
response = client.embed(
    model="embeddinggemma",
    input="RAG combines document retrieval with language models."
)

embedding = response["embeddings"][0]

print("Embedding successful!")
print("Embedding dimensions:", len(embedding))

Embedding successful!
Embedding dimensions: 768


In [31]:

import chromadb
import uuid

# Initialize ChromaDB
chroma_client = chromadb.Client()

# Create collection
try:
    chroma_client.delete_collection("college_rag")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="college_rag",
    metadata={"hnsw:space": "cosine"}
)

print("ChromaDB initialized successfully!")


ChromaDB initialized successfully!


In [32]:

from pypdf import PdfReader
from google.colab import files

def extract_text(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""

    for page in reader.pages:
        content = page.extract_text()
        if content:
            text += content + "\n"

    return text


def split_into_chunks(text, chunk_size=800, overlap=150):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


print("PDF processing functions are ready!")


PDF processing functions are ready!


In [33]:

uploaded = files.upload()

pdf_path = next(iter(uploaded.keys()))

text = extract_text(pdf_path)

chunks = split_into_chunks(text)

print("PDF uploaded successfully!")
print("Total characters:", len(text))
print("Total chunks:", len(chunks))

print("\nDocument preview:")
print(chunks[0][:500] if chunks else "No readable text found")


Saving Syllabus.pdf to Syllabus.pdf
PDF uploaded successfully!
Total characters: 144953
Total chunks: 224

Document preview:
  1 
 
1 
 
 
Probability, Distributions and Statistics 
Course Code 1BMATCS301 Scheme 2025 
Type of Course ASC Semester 3 
Teaching Hours/Week (L:T:P)   3:2:0 CIE Marks 50 
Total Hours of Pedagogy per semester 
L :T:P:SL&TW:TH      
42:28:0:50:120 SEE Marks 50 
Credits 4 Total Marks 100 
Examination type (SEE) Theory Exam Hours 03 
Course outcome (Course Skill Set) 
 
At the end of the course, the student will be able to: 
1. Apply the concepts of modular arithmetic, congruences, linear Diophan


In [36]:

import uuid

# Check whether PDF chunks exist
if not chunks:
    raise ValueError("No PDF chunks found. Please process your syllabus first.")

print("Total chunks:", len(chunks))

# Generate embeddings in batches
all_embeddings = []

for i in range(0, len(chunks), 16):

    batch = chunks[i:i + 16]

    response = client.embed(
        model="embeddinggemma",
        input=batch
    )

    batch_embeddings = response["embeddings"]

    all_embeddings.extend(batch_embeddings)

    print(
        f"Processed {min(i + 16, len(chunks))}/{len(chunks)} chunks"
    )

# Verify embedding count
assert len(all_embeddings) == len(chunks)

# Create unique IDs
ids = [str(uuid.uuid4()) for _ in chunks]

# Store in ChromaDB
collection.add(
    ids=ids,
    documents=chunks,
    embeddings=all_embeddings
)

print("\nSUCCESS!")
print("Total documents stored:", collection.count())
print("Embedding dimensions:", len(all_embeddings[0]))


Total chunks: 224


ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download

In [37]:
import subprocess
import time
import requests

# Start Ollama server
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Wait until server is available
for i in range(30):
    try:
        r = requests.get(
            "http://127.0.0.1:11434/api/tags",
            timeout=3
        )

        if r.status_code == 200:
            print("Ollama server restarted successfully!")
            print(r.json())
            break

    except requests.exceptions.RequestException:
        time.sleep(2)

else:
    raise RuntimeError("Ollama server could not start")

Ollama server restarted successfully!
{'models': [{'name': 'embeddinggemma:latest', 'model': 'embeddinggemma:latest', 'modified_at': '2026-11-26T03:50:35.769351398Z', 'size': 621875917, 'digest': '85462619ee721b466c5927d109d4cb765861907d5417b9109caebc4e614679f1', 'details': {'parent_model': 'embeddinggemma:300m-bf16', 'format': 'gguf', 'family': 'gemma3', 'families': ['gemma3'], 'parameter_size': '307.58M', 'quantization_level': 'BF16', 'context_length': 2048, 'embedding_length': 768, 'runner': 'ggml'}, 'capabilities': ['embedding']}, {'name': 'gemma3:1b', 'model': 'gemma3:1b', 'modified_at': '2026-11-26T03:50:04.175081894Z', 'size': 806067407, 'digest': '97558784bdc9175075a72655bf360b15da17df6b3b9e5182af9f8c0c8e58ba7c', 'details': {'parent_model': 'dhiltgen/gemma3:1b-it-q4_K_M-llamacpp', 'format': 'gguf', 'family': 'gemma3', 'families': ['gemma3'], 'parameter_size': '999.89M', 'quantization_level': 'Q4_K_M', 'context_length': 32768, 'embedding_length': 1152, 'runner': 'llamacpp'}, 'ca

In [38]:
import ollama

client = ollama.Client(
    host="http://127.0.0.1:11434",
    timeout=300
)

print("Ollama client connected!")

Ollama client connected!


In [39]:
test = client.embed(
    model="embeddinggemma",
    input="VTU syllabus and engineering subjects"
)

print("Embedding successful!")
print("Dimensions:", len(test["embeddings"][0]))

Embedding successful!
Dimensions: 768


In [40]:
import time

all_embeddings = []

for i in range(0, len(chunks), 4):

    batch = chunks[i:i+4]

    for attempt in range(3):
        try:
            result = client.embed(
                model="embeddinggemma",
                input=batch
            )

            all_embeddings.extend(result["embeddings"])
            break

        except Exception as e:
            print("Retrying batch:", i, e)

            time.sleep(5)

    else:
        raise RuntimeError(
            f"Embedding failed at chunk {i}. Please restart Ollama."
        )

    print(
        f"Completed: {len(all_embeddings)}/{len(chunks)}"
    )

print("All embeddings generated!")

Completed: 4/224
Completed: 8/224
Completed: 12/224
Completed: 16/224
Completed: 20/224
Completed: 24/224
Completed: 28/224
Completed: 32/224
Completed: 36/224
Completed: 40/224
Completed: 44/224
Completed: 48/224
Completed: 52/224
Completed: 56/224
Completed: 60/224
Completed: 64/224
Completed: 68/224
Completed: 72/224
Completed: 76/224
Completed: 80/224
Completed: 84/224
Completed: 88/224
Completed: 92/224
Completed: 96/224
Completed: 100/224
Completed: 104/224
Completed: 108/224
Completed: 112/224
Completed: 116/224
Completed: 120/224
Completed: 124/224
Completed: 128/224
Completed: 132/224
Completed: 136/224
Completed: 140/224
Completed: 144/224
Completed: 148/224
Completed: 152/224
Completed: 156/224
Completed: 160/224
Completed: 164/224
Completed: 168/224
Completed: 172/224
Completed: 176/224
Completed: 180/224
Completed: 184/224
Completed: 188/224
Completed: 192/224
Completed: 196/224
Completed: 200/224
Completed: 204/224
Completed: 208/224
Completed: 212/224
Completed: 216/224


In [41]:
import uuid

collection.add(
    ids=[str(uuid.uuid4()) for _ in chunks],
    documents=chunks,
    embeddings=all_embeddings
)

print("Successfully stored VTU syllabus!")
print("Total vectors:", collection.count())

Successfully stored VTU syllabus!
Total vectors: 224


In [46]:


def rag_answer(question):

    query_result = client.embed(
        model="embeddinggemma",
        input=question
    )

    query_embedding = query_result["embeddings"][0]

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=6
    )

    retrieved_chunks = results["documents"][0]

    context = "\n\n".join(retrieved_chunks)

    prompt = f"""
    You are an expert VTU academic syllabus assistant.

    Answer the student's question accurately using
    the supplied syllabus context.

    IMPORTANT RULES:
    1. Answer only what the student asks.
    2. Do not mix subject names with module topics.
    3. Do not include textbook references unless asked.
    4. Do not repeat information.
    5. Organize answers using clear headings and bullet points.
    6. If the exact semester information is unavailable,
       clearly mention that.
    7. Never invent subject names or syllabus details.

    SYLLABUS CONTEXT:
    {context}

    STUDENT QUESTION:
    {question}

    Give a clear and concise answer.
    """

    response = client.chat(
        model="gemma3:1b",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response["message"]["content"]


In [47]:
print(rag_answer(
    "List only the subject names in the first semester."
))


Here are the subject names from the first semester, according to the syllabus context:

*   Learning Activity-1
*   Learning Activity-2
*   Learning Activity-3
*   Learning Activity-4
*   Learning Activity-5
*   Learning Activity-6
*   Learning Activity-7
*   Learning Activity-8
*   Learning Activity-9
*   Learning Activity-10
*   Learning Activity-11
*   Learning Activity-12
*   Learning Activity-13
*   Learning Activity-14
*   Learning Activity-15
*   Learning Activity-16
*   Learning Activity-17
*   Learning Activity-18
*   Learning Activity-19
*   Learning Activity-20


In [48]:
print(rag_answer(
    "Explain the topics covered in Module 1 of Operating Systems."
))

Okay, here's a concise explanation of the topics covered in Module 1 of the Operating Systems syllabus, based on the provided context:

**Module 1: Introduction to Operating Systems**

Here's a breakdown of the key topics covered in Module 1:

1.  **Introduction to Operating Systems:** This section provides a foundational overview of what operating systems are, their roles in computer systems, and the overall system architecture.

2.  **System Structures:** It explains the structure of computer systems, including the roles of the operating system, its components (like the CPU, memory, and I/O devices), and the overall system organization.

3.  **Operating System Structure:** It details the structure of an operating system, covering its components and their functions.

4.  **System Services:** Introduces the various services provided by the operating system, like user interface, system calls, types of system calls, system programs, and the design and implementation of operating systems.

In [49]:
print(rag_answer(
    "What textbooks are recommended for the first semester?"
))

Here are the recommended textbooks for the first semester of “Higher Engineering Mathematics” ( Khanna Publishers, 2021), based on the syllabus context:

**Recommended Textbooks for the First Semester:**

*   **“Higher Engineering Mathematics” by S. Grewal** ( Khanna Publishers, 2021) – This is the primary textbook for the entire course and is crucial for foundational understanding.
*   **“Mathematical Statistics with Applications” by Irwin Miller & Marylees Miller** (Pearson, 2014) – This book provides a solid introduction to statistical concepts relevant to engineering.
*   **“Mathematical Statistics with Applications” by Wes McKinney** (Pearson, 2022) – A more practical approach to statistical methods.




In [50]:
!pip -q install gradio

In [51]:

import gradio as gr
import uuid

# -----------------------------
# PDF Upload Function
# -----------------------------

def upload_pdf(pdf_file):

    global collection

    if pdf_file is None:
        return "Please upload a PDF document."

    try:
        text = extract_text(pdf_file)

        if not text.strip():
            return "No readable text found in PDF."

        new_chunks = split_into_chunks(text)

        if not new_chunks:
            return "No text chunks created."

        # Generate embeddings
        embeddings = []

        for i in range(0, len(new_chunks), 4):

            batch = new_chunks[i:i+4]

            result = client.embed(
                model="embeddinggemma",
                input=batch
            )

            embeddings.extend(result["embeddings"])

        # Replace old collection
        try:
            chroma_client.delete_collection("college_rag")
        except Exception:
            pass

        collection = chroma_client.create_collection(
            name="college_rag",
            metadata={"hnsw:space": "cosine"}
        )

        # Store document chunks
        collection.add(
            ids=[str(uuid.uuid4()) for _ in new_chunks],
            documents=new_chunks,
            embeddings=embeddings
        )

        return (
            "SUCCESS! VTU syllabus uploaded and processed.\n"
            f"Total chunks indexed: {len(new_chunks)}"
        )

    except Exception as e:
        return f"Upload error: {str(e)}"


# -----------------------------
# Chat Function
# -----------------------------

def chat_with_vtu(message, history):

    if not message.strip():
        return "Please enter a question."

    if collection.count() == 0:
        return "Please upload and process your syllabus first."

    try:
        return rag_answer(message)

    except Exception as e:
        return f"Chat error: {str(e)}"


# -----------------------------
# Gradio Interface
# -----------------------------

with gr.Blocks(
    theme=gr.themes.Soft(),
    title="VTU Syllabus Assistant"
) as demo:

    gr.Markdown(
        """
        # 🎓 VTU Syllabus Assistant

        ### Your AI-powered academic companion

        Ask questions about your VTU syllabus
        using Retrieval Augmented Generation.

        **Powered by Ollama | Gemma | EmbeddingGemma | ChromaDB**
        """
    )

    with gr.Row():

        with gr.Column(scale=1):

            gr.Markdown("### 📄 Upload Syllabus")

            pdf_input = gr.File(
                label="Upload VTU PDF",
                file_types=[".pdf"],
                type="filepath"
            )

            upload_button = gr.Button(
                "Process Syllabus",
                variant="primary"
            )

            upload_status = gr.Textbox(
                label="Processing Status",
                lines=3
            )

        with gr.Column(scale=2):

            gr.Markdown("### 💬 Chat with your syllabus")

            chatbot = gr.ChatInterface(
                fn=chat_with_vtu,
                title="Ask Your VTU Assistant",
                description=(
                    "Ask about subjects, modules, "
                    "course outcomes and textbooks."
                ),
                examples=[
                    "What subjects are included in the first semester?",
                    "Explain the Operating Systems syllabus.",
                    "What are the recommended textbooks?",
                    "What topics are covered in Module 1?"
                ]
            )

    upload_button.click(
        fn=upload_pdf,
        inputs=pdf_input,
        outputs=upload_status
    )

    gr.Markdown(
        """
        ---
        **VTU Syllabus Assistant | RAG-Based Academic Chatbot**

        Built using Python, Ollama, Gemma, ChromaDB and Gradio.
        """
    )


demo.launch(share=True)


/tmp/ipykernel_2514/144189395.py:90: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ba8c2da14aa88de724.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
